# Fine-tuning with LoRA for Text Generation

## Learning Objectives

In this notebook, you will learn:
- What fine-tuning is and when to use it
- The difference between full fine-tuning and parameter-efficient methods
- How LoRA (Low-Rank Adaptation) works mathematically and practically
- How to fine-tune GPT-2 on custom data using LoRA
- How to save, load, and deploy tiny LoRA adapters
- How to compare model performance before and after fine-tuning
- Best practices for fine-tuning with limited resources

**What is LoRA?** LoRA freezes the original model weights and injects small trainable matrices into attention layers. Instead of updating millions of parameters, you train only thousands -- achieving comparable quality at a fraction of the cost.

## Prerequisites

| Requirement | Minimum | Recommended |
|------------|---------|-------------|
| RAM | 8GB | 16GB |
| GPU | Not required (slow) | 8GB+ VRAM |
| Python | 3.10+ | 3.11 |
| Storage | 5GB free | 10GB free |
| Time | 30-60 min on CPU | 5-10 min on GPU |

| Model Option | Model Name | Parameters | Size | Notes |
|-------------|------------|-----------|------|-------|
| **Base Model** | gpt2 | 124M | ~500MB | CPU-compatible, recommended for this notebook |

**Note**: This notebook is designed to work on CPU, unlike Notebook 04 (Unsloth) which requires a GPU. Training on CPU is slower but fully functional.

## Expected Behaviors

**Dataset Download**:
- TinyStories dataset downloads automatically (~300MB for 5000 examples)
- Progress bar shows download status
- Cached locally after first download

**Model Loading**:
- Base model (GPT-2 small): ~500MB download on first run
- LoRA adds minimal overhead (<10MB)
- GPU: loads in 5-10 seconds, CPU: 10-20 seconds

**Training**:
- GPU (RTX 4080): 5-10 minutes for 1000 steps
- CPU: 30-60+ minutes (functional but slow)
- Loss should decrease from ~3-4 to ~2-3 after 1 epoch

**After Fine-tuning**:
- Model generates story-like text with simpler vocabulary
- Outputs become more coherent and child-friendly
- LoRA adapter saves as ~3-5MB (vs ~500MB full model)

## Overview

### What is Fine-tuning?

**Fine-tuning** adapts a pre-trained model to your specific use case by continuing training on your custom dataset.

**When to fine-tune**:
- Model does not understand your domain (medical, legal, gaming)
- You want a specific writing style
- Pre-trained models are not good enough for your task
- You have relevant training data (1000+ examples recommended)

**When NOT to fine-tune**:
- Pre-trained model already works well
- You have very little data (<100 examples)
- You only need basic text generation

### How LoRA Works

In standard fine-tuning, you update the full weight matrix $W \in \mathbb{R}^{d \times k}$. LoRA instead decomposes the update into two smaller matrices:

$$W' = W + \Delta W = W + BA$$

where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$, with rank $r \ll \min(d, k)$.

**Example**: For GPT-2's attention layer with $d = k = 768$:
- Full fine-tuning: $768 \times 768 = 589,824$ parameters
- LoRA with $r = 8$: $(768 \times 8) + (8 \times 768) = 12,288$ parameters (2% of full)

The scaling factor $\alpha$ controls the magnitude of the LoRA update:

$$W' = W + \frac{\alpha}{r} \cdot BA$$

## Setup and Installation

If you have not installed PEFT yet, uncomment and run the pip install line below, then restart the kernel.

In [ ]:
# Uncomment to install PEFT if needed:
# !pip install peft accelerate

import sys, time, random, os, warnings
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    set_seed,
)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from datasets import load_dataset

warnings.filterwarnings("ignore")

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Device: {device}")

## Part 1: Load Base Model and Dataset

We start by loading GPT-2 and establishing a baseline for text generation quality before any fine-tuning.

### 1.1 Load Pre-trained Model

GPT-2 small has 124M parameters and is small enough to fine-tune on CPU.

In [ ]:
MODEL_NAME = "gpt2"

try:
    print(f"Loading model: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
    )
    base_model.to(device)
    base_model.config.pad_token_id = tokenizer.eos_token_id

    total_params = sum(p.numel() for p in base_model.parameters())
    print(f"Model loaded: {total_params / 1e6:.1f}M parameters")
    print(f"Device: {device}")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Ensure you have internet access and transformers installed.")

### 1.2 Test Base Model (Before Fine-tuning)

Let's see how GPT-2 generates text before fine-tuning to establish a baseline.

In [ ]:
def generate_text(
    model, prompt: str, max_length: int = 100, num_return: int = 3
) -> list[str]:
    """Generate text continuations from a prompt.

    Args:
        model: The language model to use for generation.
        prompt: Input text to continue.
        max_length: Maximum total sequence length.
        num_return: Number of sequences to generate.

    Returns:
        List of generated text strings.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_return_sequences=num_return,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    return [tokenizer.decode(out, skip_special_tokens=True) for out in outputs]


test_prompts = [
    "Once upon a time, there was a little girl named",
    "The brave knight went to the castle and",
]

print("BASE MODEL OUTPUT (before fine-tuning)")
print("=" * 70)
for prompt in test_prompts:
    print(f"\nPrompt: {prompt}")
    outputs = generate_text(base_model, prompt, max_length=60, num_return=1)
    print(f"Output: {outputs[0]}")
    print("-" * 70)

Notice how the base model generates general-purpose text that does not sound like a children's story. After fine-tuning on TinyStories, the model should produce simpler, story-like outputs.

### 1.3 Load and Prepare Dataset

TinyStories is a dataset of short children's stories with simple vocabulary -- ideal for demonstrating fine-tuning effects.

In [ ]:
try:
    print("Loading dataset: roneneldan/TinyStories")
    dataset = load_dataset("roneneldan/TinyStories", split="train[:5000]")
    print(f"Dataset loaded: {len(dataset)} examples")
    print(f"\nSample story (first 200 chars):")
    print(dataset[0]["text"][:200])
except Exception as e:
    print(f"Error loading dataset: {e}")
    print("Ensure you have internet access and the datasets library installed.")

### 1.4 Tokenize Dataset

We tokenize the stories into fixed-length sequences for efficient training.

In [ ]:
MAX_LENGTH = 512


def tokenize_function(examples: dict) -> dict:
    """Tokenize texts for causal language modeling.

    Args:
        examples: Batch of examples with a 'text' field.

    Returns:
        Tokenized batch with input_ids, attention_mask, and labels.
    """
    outputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors=None,
    )
    outputs["labels"] = outputs["input_ids"].copy()
    return outputs


print("Tokenizing dataset...")
tokenized = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)

split = tokenized.train_test_split(test_size=0.1, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train examples: {len(train_dataset)}")
print(f"Eval examples: {len(eval_dataset)}")

## Part 2: Configure LoRA

Now we configure LoRA to add small trainable adapters to GPT-2's attention layers. The original weights stay frozen.

In [ ]:
LORA_RANK = 8
LORA_ALPHA = 32
LORA_DROPOUT = 0.1

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["c_attn", "c_proj"],
)

model = get_peft_model(base_model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

config_df = pd.DataFrame([
    {"Parameter": "LoRA rank (r)", "Value": LORA_RANK},
    {"Parameter": "LoRA alpha", "Value": LORA_ALPHA},
    {"Parameter": "LoRA dropout", "Value": LORA_DROPOUT},
    {"Parameter": "Target modules", "Value": "c_attn, c_proj"},
    {"Parameter": "Total parameters", "Value": f"{total / 1e6:.1f}M"},
    {"Parameter": "Trainable parameters", "Value": f"{trainable / 1e6:.2f}M"},
    {"Parameter": "Trainable %", "Value": f"{100 * trainable / total:.2f}%"},
])

print("LoRA Configuration")
config_df

With LoRA rank 8, we are training less than 1% of the total parameters. This is what makes LoRA so memory-efficient.

## Part 3: Fine-tune the Model

We now train the LoRA adapters on TinyStories. The frozen base model weights are unchanged -- only the small LoRA matrices are updated.

### 3.1 Training Configuration

In [ ]:
BATCH_SIZE = 4 if device.type == "cuda" else 1
LEARNING_RATE = 2e-4
NUM_EPOCHS = 1

training_args = TrainingArguments(
    output_dir="./lora_tinystories",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=500,
    fp16=(device.type == "cuda"),
    report_to="none",
    seed=SEED,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

train_config_df = pd.DataFrame([
    {"Parameter": "Batch size", "Value": str(BATCH_SIZE)},
    {"Parameter": "Learning rate", "Value": str(LEARNING_RATE)},
    {"Parameter": "Epochs", "Value": str(NUM_EPOCHS)},
    {"Parameter": "Warmup steps", "Value": "100"},
    {"Parameter": "Weight decay", "Value": "0.01"},
    {"Parameter": "FP16", "Value": str(device.type == "cuda")},
])

print("Training Configuration")
train_config_df

### 3.2 Create Trainer and Start Training

Training will take 5-10 minutes on GPU or 30-60+ minutes on CPU. The loss should decrease steadily.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("Starting training...")
print("=" * 70)
if device.type == "cpu":
    print("Training on CPU -- this will be slow (30-60+ minutes).")
    print("For faster training, see Notebook 04 (Unsloth, GPU required).")
print()

start = time.perf_counter()
train_result = trainer.train()
train_time = time.perf_counter() - start

print(f"\nTraining completed in {train_time:.1f}s ({train_time / 60:.1f} min)")
print(f"Final training loss: {train_result.training_loss:.4f}")

## Part 4: Test Fine-tuned Model

Let's compare the fine-tuned model's outputs to the base model. We expect the fine-tuned model to generate simpler, story-like text.

In [ ]:
test_prompts = [
    "Once upon a time, there was a little girl named",
    "The brave knight went to the castle and",
    "In the forest, the animals were playing when",
]

rows = []
for prompt in test_prompts:
    base_model.to(device)
    base_output = generate_text(base_model, prompt, max_length=60, num_return=1)[0]
    fine_output = generate_text(model, prompt, max_length=60, num_return=1)[0]
    rows.append({
        "Prompt": prompt[:40] + "...",
        "Base Model": base_output[len(prompt):].strip()[:80],
        "Fine-tuned": fine_output[len(prompt):].strip()[:80],
    })

comparison_df = pd.DataFrame(rows)
print("Before vs After Fine-tuning")
comparison_df

The fine-tuned model should produce text that reads more like a children's story -- simpler vocabulary, narrative structure, and character-driven plot.

## Part 5: Save and Load LoRA Adapters

One of LoRA's biggest advantages is that adapters are tiny. The full GPT-2 model is ~500MB, but LoRA adapters are only a few MB.

### 5.1 Save Adapters

In [ ]:
ADAPTER_PATH = "./tinystories_lora_adapter"

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

adapter_size = sum(
    os.path.getsize(os.path.join(ADAPTER_PATH, f))
    for f in os.listdir(ADAPTER_PATH)
    if os.path.isfile(os.path.join(ADAPTER_PATH, f))
)

size_df = pd.DataFrame([
    {"Component": "Full GPT-2 model", "Size": "~500 MB"},
    {"Component": "LoRA adapter", "Size": f"{adapter_size / 1024 / 1024:.1f} MB"},
    {"Component": "Savings", "Size": f"{500 / max(adapter_size / 1024 / 1024, 0.1):.0f}x smaller"},
])

print(f"Adapter saved to: {ADAPTER_PATH}")
size_df

### 5.2 Load Adapters

This demonstrates how to load saved LoRA adapters onto a fresh base model -- the pattern you would use in production.

In [ ]:
def load_lora_model(base_model_name: str, adapter_path: str) -> tuple:
    """Load a base model with LoRA adapters applied.

    Args:
        base_model_name: HuggingFace model identifier for the base model.
        adapter_path: Local path to saved LoRA adapter weights.

    Returns:
        Tuple of (model, tokenizer) with adapters loaded.
    """
    base = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
    )
    loaded = PeftModel.from_pretrained(base, adapter_path)
    loaded.to(device)
    tok = AutoTokenizer.from_pretrained(base_model_name)
    tok.pad_token = tok.eos_token
    return loaded, tok


model_reload, _ = load_lora_model(MODEL_NAME, ADAPTER_PATH)

test_output = generate_text(model_reload, "Once upon a time", max_length=50, num_return=1)
print("Loaded model output:")
print(test_output[0])

## Part 6: Evaluation and Analysis

Let's quantify the improvement from fine-tuning using validation loss and perplexity.

### 6.1 Quantitative Evaluation

In [ ]:
print("Evaluating on validation set...")
eval_results = trainer.evaluate()

eval_loss = eval_results["eval_loss"]
perplexity = torch.exp(torch.tensor(eval_loss)).item()

eval_df = pd.DataFrame([
    {"Metric": "Validation loss", "Value": f"{eval_loss:.4f}"},
    {"Metric": "Perplexity", "Value": f"{perplexity:.2f}"},
    {"Metric": "Training loss (final)", "Value": f"{train_result.training_loss:.4f}"},
    {"Metric": "Training time", "Value": f"{train_time:.1f}s"},
])

print("Evaluation Results")
eval_df

### 6.2 Side-by-Side Comparison

A final detailed comparison on a single prompt with multiple generations.

In [ ]:
COMPARISON_PROMPT = "The little mouse found a piece of cheese and"

base_model.to(device)
base_outputs = generate_text(base_model, COMPARISON_PROMPT, max_length=80, num_return=3)
fine_outputs = generate_text(model, COMPARISON_PROMPT, max_length=80, num_return=3)

rows = []
for i, (b, f) in enumerate(zip(base_outputs, fine_outputs), 1):
    rows.append({
        "#": i,
        "Base Model": b[len(COMPARISON_PROMPT):].strip()[:100],
        "Fine-tuned": f[len(COMPARISON_PROMPT):].strip()[:100],
    })

print(f"Prompt: {COMPARISON_PROMPT}")
pd.DataFrame(rows)

## Performance Benchmarking

Let's measure inference speed for the base model vs the LoRA model.

In [ ]:
def benchmark_generation(model, prompt: str, n_runs: int = 5, max_length: int = 60) -> dict:
    """Benchmark text generation speed.

    Args:
        model: The language model to benchmark.
        prompt: Input text for generation.
        n_runs: Number of runs to average over.
        max_length: Maximum sequence length.

    Returns:
        Dict with timing statistics.
    """
    times = []
    for _ in range(n_runs):
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        start = time.perf_counter()
        with torch.no_grad():
            model.generate(**inputs, max_length=max_length, pad_token_id=tokenizer.eos_token_id)
        times.append(time.perf_counter() - start)
    return {"mean_ms": np.mean(times) * 1000, "std_ms": np.std(times) * 1000}


bench_prompt = "Once upon a time"

base_model.to(device)
base_bench = benchmark_generation(base_model, bench_prompt)
lora_bench = benchmark_generation(model, bench_prompt)

bench_df = pd.DataFrame([
    {"Model": "GPT-2 (base)", "Mean (ms)": f"{base_bench['mean_ms']:.1f}", "Std (ms)": f"{base_bench['std_ms']:.1f}"},
    {"Model": "GPT-2 + LoRA", "Mean (ms)": f"{lora_bench['mean_ms']:.1f}", "Std (ms)": f"{lora_bench['std_ms']:.1f}"},
])

print(f"Inference Benchmark ({device})")
bench_df

LoRA adds minimal inference overhead since the adapter matrices are small. In production, you can merge the LoRA weights into the base model for zero overhead.

## Error Analysis

Let's test the fine-tuned model on prompts outside its training distribution to understand its limitations.

In [ ]:
def analyze_failure_modes(model) -> pd.DataFrame:
    """Test the model on out-of-distribution prompts.

    Args:
        model: The fine-tuned language model.

    Returns:
        DataFrame with test cases, outputs, and observations.
    """
    test_cases = [
        ("Factual question", "The capital of France is"),
        ("Technical writing", "To install Python, first"),
        ("Non-English", "Il etait une fois un petit"),
        ("Code generation", "def fibonacci(n):"),
        ("Very short prompt", "The"),
    ]

    rows = []
    for category, prompt in test_cases:
        output = generate_text(model, prompt, max_length=50, num_return=1)[0]
        generated = output[len(prompt):].strip()[:80]
        rows.append({"Category": category, "Prompt": prompt, "Output": generated})

    return pd.DataFrame(rows)


print("Error Analysis: Out-of-Distribution Prompts")
print("The model was fine-tuned on children's stories (TinyStories).")
print("Prompts outside this domain reveal domain collapse.\n")
analyze_failure_modes(model)

**Observations**: After fine-tuning on TinyStories, the model tends to steer all outputs toward story-like text, even when prompted with factual questions or code. This is expected -- the model has been adapted to a narrow domain. In production, you would use the LoRA adapter only for story-generation tasks and fall back to the base model for other uses.

## Best Practices and Tips

### Hyperparameter Guidelines

In [ ]:
hp_df = pd.DataFrame([
    {"Parameter": "LoRA rank (r)", "Low": "1-4", "Medium (recommended)": "8-16", "High": "32-64", "Effect": "Higher = more capacity, more memory"},
    {"Parameter": "Learning rate", "Low": "<1e-5", "Medium (recommended)": "1e-4 to 3e-4", "High": ">5e-4", "Effect": "Too high = diverge, too low = slow"},
    {"Parameter": "LoRA alpha", "Low": "8", "Medium (recommended)": "16-32", "High": "64+", "Effect": "Scales LoRA contribution"},
    {"Parameter": "Dropout", "Low": "0.0", "Medium (recommended)": "0.05-0.1", "High": "0.2+", "Effect": "Regularization for small datasets"},
    {"Parameter": "Epochs", "Low": "1", "Medium (recommended)": "2-3", "High": "5+", "Effect": "More = better fit, risk overfitting"},
])

print("Hyperparameter Guidelines")
hp_df

### Common Issues and Solutions

**CUDA out of memory**: Reduce batch size, reduce max_length, or use CPU.

**Loss not decreasing**: Check learning rate (try 1e-4 to 3e-4), verify data quality, ensure tokenization is correct.

**Outputs unchanged after training**: Increase epochs or LoRA rank. Verify the LoRA adapters are actually being trained (check trainable parameter count).

**Overfitting (validation loss increasing)**: Reduce epochs, increase dropout, use more training data.

### When to Use Full Fine-tuning vs LoRA

| Criterion | Full Fine-tuning | LoRA |
|-----------|-----------------|------|
| GPU memory | 16GB+ for GPT-2 | 8GB sufficient |
| Training speed | Slow | 3-5x faster |
| Model size on disk | Full copy (~500MB) | Adapter only (~3-5MB) |
| Multiple variants | One model per variant | Swap adapters |
| Dataset size | Large (10k+) | Small-medium (1k-10k) |
| Quality ceiling | Highest | Near-equivalent |

## Exercises

1. **Different Dataset**: Fine-tune GPT-2 on a different dataset (e.g., `imdb` for movie reviews, `squad` for Q&A). Compare outputs.

2. **Hyperparameter Tuning**: Experiment with different LoRA ranks (r=4, r=16, r=32). How does this affect training time and output quality?

3. **Target Modules**: Try adding more target modules (e.g., `c_fc`) to the LoRA config. Does it improve quality?

4. **Longer Training**: Train for 3 epochs instead of 1. Does the model improve or overfit?

5. **Merge Adapters**: Use `model.merge_and_unload()` to merge LoRA weights into the base model. Compare inference speed before and after merging.

## Key Takeaways

**Fine-tuning**:
- Adapts pre-trained models to your specific use case
- Requires custom dataset (1000+ examples recommended)
- Much faster than training from scratch

**LoRA**:
- Trains only 0.1-1% of parameters via low-rank matrix decomposition
- 3-5x faster and much less memory than full fine-tuning
- Tiny adapters (MBs vs GBs) that can be swapped at runtime
- Near-equivalent quality to full fine-tuning for most tasks

**Practical considerations**:
- Fine-tuning causes domain specialization (good for target task, may degrade on others)
- LoRA rank controls the capacity/efficiency tradeoff
- Always compare before and after to verify improvement

## Next Steps & Resources

**Next Notebooks**:
- **Notebook 04**: Unsloth fine-tuning (2-5x faster, GPU required)
- **Notebook 10**: Ollama integration for local LLM deployment

**Papers**:
- [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)
- [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314)

**Libraries**:
- [PEFT Documentation](https://huggingface.co/docs/peft)
- [TRL Documentation](https://huggingface.co/docs/trl)
- [HuggingFace Fine-tuning Guide](https://huggingface.co/docs/transformers/training)